# Text Cleanup Exploration

Explore OCR cleanup behavior for mixed English/Japanese martial arts text. The important rule is preservation: Japanese characters, macrons, and martial arts punctuation must survive cleanup.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from utils.text.text_utils import LanguageDetector, TextCleaner, TextStatistics
from martial_arts_ocr.pipeline.extraction_adapters import text_region_from_extraction

sample_text = """Donn Draeger studied classical bujutsu.  
武道とは何か。
柔術 and Daitō-ryū Aiki-jūjutsu require careful OCR cleanup.
Terms: koryū, budō, ō, ū, —, ・, 「形」
■


rn may be an OCR error in some English words.
"""

In [ ]:
cleaner = TextCleaner()
cleaned, stats = cleaner.clean_text(sample_text, aggressive=False)
print("Before:\n", sample_text)
print("After:\n", cleaned)
print(stats.to_dict())

In [ ]:
must_survive = ["武道", "柔術", "koryū", "budō", "Daitō-ryū", "ō", "ū", "—", "・", "「", "」"]
missing = [token for token in must_survive if token not in cleaned]
print("Missing preservation tokens:", missing)
assert not missing

In [ ]:
detector = LanguageDetector()
segments = detector.segment_by_language(cleaned)
for segment in segments:
    print(segment.to_dict())

stats_report = TextStatistics().calculate_statistics(cleaned)
print(stats_report)

In [ ]:
text_region = text_region_from_extraction({
    "region_id": "mixed_text_001",
    "text": cleaned,
    "language": "mixed",
    "confidence": 0.90,
    "bbox": (20, 30, 620, 210),
})
text_region.to_dict()